# 04 — Fusion Layer (Ensemble Combination of Isolation Forest, Random Forest, Autoencoder)

Combines the three already-trained ensemble members' validation-set outputs into a single
fused decision, and answers the central open question raised by `03_autoencoder.ipynb`:
Isolation Forest and Autoencoder's per-category recall correlate at **0.925** (val) and
**0.977** (LOAO) — their blind spots almost perfectly overlap. Does adding Autoencoder as a
third ensemble member measurably improve detection over a simpler Isolation Forest +
Random Forest combination, or is Random Forest + Autoencoder (dropping the redundant
Isolation Forest) just as good?

This notebook does **not** retrain any of the three base models for the main validation-set
analysis — it only loads and combines their already-saved outputs
(`cicids2017_isoforest_scores_val.parquet`, `cicids2017_rf_scores_val.parquet`,
`cicids2017_autoencoder_scores_val.parquet`). The zero-day (LOAO) section in §8 is the one
exception: none of the three base notebooks saved row-level LOAO predictions (only aggregate
per-category recall/FPR), so answering the zero-day fusion question requires retraining all
three models per fold here, using each model's own already-established hyperparameters
verbatim (documented alongside each in §1's config) — a deliberate scope exception, decided
with the user rather than assumed.

Known val-set results from the three base models, for reference:
- **Isolation Forest** (`contamination=0.20`): Precision 0.529 / Recall 0.556 / F1 0.542 /
  AUC-ROC 0.729. Strong on flood-style DoS/DDoS, near-zero on FTP-/SSH-Patator/PortScan/Bot.
- **Random Forest** (SMOTE-resampled, multi-class): binary roll-up Precision 0.9955 / Recall
  0.9958 / F1 0.9957 / AUC-ROC 0.9998. Excellent in-distribution; collapses on categories it
  never saw in LOAO zero-day testing.
- **Autoencoder** (BENIGN-only, 99th-percentile threshold): Precision 0.914 / Recall 0.521 /
  F1 0.664 / AUC-ROC 0.864. Blind-spot profile closely mirrors Isolation Forest's.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

print("torch version:", torch.__version__, " | cuda available:", torch.cuda.is_available())

In [ ]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PROCESSED_DIR = ROOT / "data" / "processed"
DATA_FEATURES_DIR = ROOT / "data" / "features"
LOAO_DIR = ROOT / "data" / "processed" / "loao"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

print("ROOT:", ROOT)

In [ ]:
RANDOM_STATE = 42
THRESHOLD_PERCENTILES = [90, 95, 99]

FUSION_WEIGHTS = {"isolation_forest": 1 / 3, "random_forest": 1 / 3, "autoencoder": 1 / 3}

RF_CONFIDENCE_THRESHOLD = 0.5

IF_N_ESTIMATORS = 100
IF_CONTAMINATION = 0.2

RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = None
RF_CLASS_WEIGHT = "balanced"
RF_N_JOBS = -1
SMOTE_FLAT_CAP = 50_000
SMOTE_TARGET_MULTIPLIER = 20
SMOTE_RANDOM_STATE = 42

AE_HIDDEN_DIMS = [64, 32]
AE_BOTTLENECK_DIM = 16
AE_ACTIVATION = "relu"
AE_LEARNING_RATE = 1e-3
AE_EPOCHS = 30
AE_BATCH_SIZE = 2048
AE_WEIGHT_DECAY = 0.0
AE_EARLY_STOPPING_PATIENCE = 5
AE_BENIGN_INTERNAL_VAL_FRAC = 0.10
AE_CHOSEN_PERCENTILE = 99
DEVICE = torch.device("cuda")

AE_CONFIG = {
    "hidden_dims": AE_HIDDEN_DIMS,
    "bottleneck_dim": AE_BOTTLENECK_DIM,
    "activation": AE_ACTIVATION,
    "learning_rate": AE_LEARNING_RATE,
    "epochs": AE_EPOCHS,
    "batch_size": AE_BATCH_SIZE,
    "weight_decay": AE_WEIGHT_DECAY,
    "early_stopping_patience": AE_EARLY_STOPPING_PATIENCE,
}

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 2. Load val-set scores and verify row alignment

`don't assume, verify`: beyond checking row counts and that each score file's `row_index`
is a plain `0..N-1` sequence, each model's own already-reported validation metrics are
recomputed here from the join and compared against the values already reported by that
model's own notebook. If the three files were misaligned relative to `val_df`, the
recomputed numbers would diverge sharply from the already-reported ones -- a much stronger
check than shape/index matching alone.

In [ ]:
val_df = pd.read_parquet(DATA_FEATURES_DIR / "cicids2017_features_val.parquet")
iso_scores = pd.read_parquet(DATA_FEATURES_DIR / "cicids2017_isoforest_scores_val.parquet")
rf_scores = pd.read_parquet(DATA_FEATURES_DIR / "cicids2017_rf_scores_val.parquet")
ae_scores = pd.read_parquet(DATA_FEATURES_DIR / "cicids2017_autoencoder_scores_val.parquet")

print("val_df       :", val_df.shape)
print("iso_scores   :", iso_scores.shape)
print("rf_scores    :", rf_scores.shape)
print("ae_scores    :", ae_scores.shape)

n = len(val_df)
assert len(iso_scores) == len(rf_scores) == len(ae_scores) == n, "row counts must match val_df exactly"

for name, df in [("iso_scores", iso_scores), ("rf_scores", rf_scores), ("ae_scores", ae_scores)]:
    assert np.array_equal(df["row_index"].to_numpy(), np.arange(n)), (
        f"{name}'s row_index is not a plain 0..N-1 sequence -- cannot assume positional alignment"
    )
print("\nRow counts match and all three score files carry a plain 0..N-1 row_index -- "
      "positional alignment confirmed structurally.")

In [ ]:
y_is_attack = (val_df["Label"] != "BENIGN").astype(int).to_numpy()

recomputed_if = {
    "precision": precision_score(y_is_attack, iso_scores["prediction"]),
    "recall": recall_score(y_is_attack, iso_scores["prediction"]),
    "f1": f1_score(y_is_attack, iso_scores["prediction"]),
    "auc_roc": roc_auc_score(y_is_attack, iso_scores["anomaly_score"]),
}
recomputed_rf = {
    "precision": precision_score(y_is_attack, rf_scores["prediction"]),
    "recall": recall_score(y_is_attack, rf_scores["prediction"]),
    "f1": f1_score(y_is_attack, rf_scores["prediction"]),
    "auc_roc": roc_auc_score(y_is_attack, rf_scores["attack_probability"]),
}
recomputed_ae = {
    "precision": precision_score(y_is_attack, ae_scores["prediction"]),
    "recall": recall_score(y_is_attack, ae_scores["prediction"]),
    "f1": f1_score(y_is_attack, ae_scores["prediction"]),
    "auc_roc": roc_auc_score(y_is_attack, ae_scores["reconstruction_error"]),
}

EXPECTED_IF = {"precision": 0.529, "recall": 0.556, "f1": 0.542, "auc_roc": 0.729}
EXPECTED_RF = {"precision": 0.9955, "recall": 0.9958, "f1": 0.9957, "auc_roc": 0.9998}
EXPECTED_AE = {"precision": 0.914, "recall": 0.521, "f1": 0.664, "auc_roc": 0.864}

alignment_check_df = pd.DataFrame({
    "isolation_forest_recomputed": recomputed_if, "isolation_forest_expected": EXPECTED_IF,
    "random_forest_recomputed": recomputed_rf, "random_forest_expected": EXPECTED_RF,
    "autoencoder_recomputed": recomputed_ae, "autoencoder_expected": EXPECTED_AE,
}).T
print()
print(alignment_check_df)

for name, recomputed, expected in [
    ("Isolation Forest", recomputed_if, EXPECTED_IF),
    ("Random Forest", recomputed_rf, EXPECTED_RF),
    ("Autoencoder", recomputed_ae, EXPECTED_AE),
]:
    for metric, expected_val in expected.items():
        assert abs(recomputed[metric] - expected_val) < 0.01, (
            f"{name}'s recomputed {metric} ({recomputed[metric]:.4f}) does not match its "
            f"already-reported value ({expected_val:.4f}) -- STOP, row alignment may be wrong"
        )
print("\nCONFIRMED: all three models' recomputed val metrics match their already-reported "
      "values (within 0.01) -- row alignment across val_df / iso_scores / rf_scores / "
      "ae_scores is correct.")

## 3. Normalize scores to a common [0, 1] scale

Isolation Forest's `anomaly_score` and the Autoencoder's `reconstruction_error` are on
unrelated, unbounded scales -- both are min-max normalized here (fit on the val set they're
being scored on, a shape-only rescaling that touches no labels). Random Forest's
`attack_probability` is already a proper `[0, 1]` probability and is used as-is.

In [ ]:
def minmax_normalize(x):
    x = np.asarray(x, dtype=np.float64)
    lo, hi = x.min(), x.max()
    if hi <= lo:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)

fusion_df = val_df[["Label"]].copy()
fusion_df["is_attack"] = y_is_attack
fusion_df["if_pred"] = iso_scores["prediction"].to_numpy()
fusion_df["rf_pred"] = rf_scores["prediction"].to_numpy()
fusion_df["ae_pred"] = ae_scores["prediction"].to_numpy()
fusion_df["if_norm"] = minmax_normalize(iso_scores["anomaly_score"])
fusion_df["ae_norm"] = minmax_normalize(ae_scores["reconstruction_error"])
fusion_df["rf_norm"] = rf_scores["attack_probability"].to_numpy()
fusion_df["rf_attack_probability"] = rf_scores["attack_probability"].to_numpy()

print("Normalized score ranges (all should be within [0, 1]):")
print(fusion_df[["if_norm", "rf_norm", "ae_norm"]].agg(["min", "max"]))
fusion_df.head()

## 4. Fusion Strategy A — weighted average of normalized scores

`fused_score = w_if * if_norm + w_rf * rf_norm + w_ae * ae_norm` (weights renormalized to
sum to 1 among whichever models are included). Threshold chosen the same way as
Isolation Forest's contamination sweep and the Autoencoder's percentile sweep: candidate
cutoffs are percentiles of the val-set BENIGN fused-score distribution, evaluated against
F1.

In [ ]:
def fuse_weighted_average(df, weights):
    total_w = sum(weights.values())
    score = np.zeros(len(df))
    if "isolation_forest" in weights:
        score += weights["isolation_forest"] / total_w * df["if_norm"].to_numpy()
    if "random_forest" in weights:
        score += weights["random_forest"] / total_w * df["rf_norm"].to_numpy()
    if "autoencoder" in weights:
        score += weights["autoencoder"] / total_w * df["ae_norm"].to_numpy()
    return score

def sweep_threshold(df, score_col, percentiles=THRESHOLD_PERCENTILES):
    benign_scores = df.loc[df["is_attack"] == 0, score_col]
    rows = []
    for pct in percentiles:
        threshold = float(np.percentile(benign_scores, pct))
        pred = (df[score_col] > threshold).astype(int)
        rows.append({
            "percentile": pct,
            "threshold": threshold,
            "precision": precision_score(df["is_attack"], pred),
            "recall": recall_score(df["is_attack"], pred),
            "f1": f1_score(df["is_attack"], pred),
        })
    return pd.DataFrame(rows)

fusion_df["weighted_avg_score"] = fuse_weighted_average(fusion_df, FUSION_WEIGHTS)
weighted_avg_sweep = sweep_threshold(fusion_df, "weighted_avg_score")
weighted_avg_auc = roc_auc_score(fusion_df["is_attack"], fusion_df["weighted_avg_score"])
weighted_avg_sweep["auc_roc"] = weighted_avg_auc
print("Strategy A (weighted average) threshold sweep:")
print(weighted_avg_sweep)

best_a = weighted_avg_sweep.loc[weighted_avg_sweep["f1"].idxmax()]
strategy_a_threshold = float(best_a["threshold"])
fusion_df["weighted_avg_pred"] = (fusion_df["weighted_avg_score"] > strategy_a_threshold).astype(int)

strategy_a_metrics = {
    "precision": precision_score(fusion_df["is_attack"], fusion_df["weighted_avg_pred"]),
    "recall": recall_score(fusion_df["is_attack"], fusion_df["weighted_avg_pred"]),
    "f1": f1_score(fusion_df["is_attack"], fusion_df["weighted_avg_pred"]),
    "auc_roc": weighted_avg_auc,
}
print(f"\nStrategy A chosen threshold: {strategy_a_threshold:.6f} ({int(best_a['percentile'])}th percentile)")
print("Strategy A (weighted average, 3-model) val metrics:", strategy_a_metrics)

## 5. Fusion Strategy B — confidence-based fallback

Flag as attack if Random Forest's own `attack_probability` exceeds `RF_CONFIDENCE_THRESHOLD`,
**or** if Isolation Forest and Autoencoder *both* already flag the row (using each model's
own already-established threshold, no extra tuning). A continuous "rule score" (fuzzy-logic
relaxation: OR → max, AND → min) is computed only to get a comparable AUC-ROC number --
the actual binary decision above never uses it.

In [ ]:
def fuse_confidence_based(df, rf_confidence_threshold=RF_CONFIDENCE_THRESHOLD,
                           use_if=True, use_ae=True):
    rf_flag = (df["rf_attack_probability"] > rf_confidence_threshold)
    if use_if and use_ae:
        unsupervised_agree = df["if_pred"].astype(bool) & df["ae_pred"].astype(bool)
    elif use_if:
        unsupervised_agree = df["if_pred"].astype(bool)
    elif use_ae:
        unsupervised_agree = df["ae_pred"].astype(bool)
    else:
        unsupervised_agree = pd.Series(False, index=df.index)
    return (rf_flag | unsupervised_agree).astype(int)

def confidence_rule_score(df, use_if=True, use_ae=True):
    rf_term = df["rf_attack_probability"].to_numpy()
    if use_if and use_ae:
        unsup_term = np.minimum(df["if_norm"].to_numpy(), df["ae_norm"].to_numpy())
    elif use_if:
        unsup_term = df["if_norm"].to_numpy()
    elif use_ae:
        unsup_term = df["ae_norm"].to_numpy()
    else:
        unsup_term = np.zeros(len(df))
    return np.maximum(rf_term, unsup_term)

fusion_df["confidence_pred"] = fuse_confidence_based(fusion_df)
fusion_df["confidence_rule_score"] = confidence_rule_score(fusion_df)

strategy_b_metrics = {
    "precision": precision_score(fusion_df["is_attack"], fusion_df["confidence_pred"]),
    "recall": recall_score(fusion_df["is_attack"], fusion_df["confidence_pred"]),
    "f1": f1_score(fusion_df["is_attack"], fusion_df["confidence_pred"]),
    "auc_roc": roc_auc_score(fusion_df["is_attack"], fusion_df["confidence_rule_score"]),
}
print(f"Strategy B (confidence-based fallback, RF_CONFIDENCE_THRESHOLD={RF_CONFIDENCE_THRESHOLD}, "
      f"3-model) val metrics:")
print(strategy_b_metrics)

### 5.1 Comparing strategies and choosing the primary one

Confidence-based fallback is preferred unless weighted averaging wins by a clear margin
(> 0.02 F1), because it composes cleanly with the LOAO per-fold retraining in §8: it reuses
each model's own already-established fold-level decision (Isolation Forest's tuned
contamination, the Autoencoder's tuned percentile, Random Forest's own probability) with no
extra cross-model score normalization or additional held-out calibration split. Weighted
averaging would need exactly that per fold -- and unlike the Autoencoder, Isolation Forest's
and Random Forest's LOAO folds don't reserve a held-out internal-validation slice to
calibrate a joint threshold against without touching the zero-day test set's own labels.

In [ ]:
strategy_comparison_df = pd.DataFrame({
    "weighted_average": strategy_a_metrics,
    "confidence_based": strategy_b_metrics,
}).T
print(strategy_comparison_df)

F1_MARGIN = 0.02
if strategy_a_metrics["f1"] > strategy_b_metrics["f1"] + F1_MARGIN:
    PRIMARY_STRATEGY = "weighted_average"
else:
    PRIMARY_STRATEGY = "confidence_based"

print(f"\nPRIMARY_STRATEGY = '{PRIMARY_STRATEGY}'")

## 6. Ablation: 2-model vs. 3-model fusion (validation set)

Using `PRIMARY_STRATEGY`, compute Isolation Forest + Random Forest (no Autoencoder),
Random Forest + Autoencoder (no Isolation Forest), and the full 3-model fusion, alongside
each base model's own already-established single-model numbers (not re-tuned here, so each
model's own reported operating point stays intact).

In [ ]:
def evaluate_combo(df, members, strategy):
    assert "random_forest" in members, "all ablation combinations in this notebook include Random Forest"
    use_if = "isolation_forest" in members
    use_ae = "autoencoder" in members

    if strategy == "weighted_average":
        weights = {k: v for k, v in FUSION_WEIGHTS.items() if k in members}
        score = fuse_weighted_average(df, weights)
        sweep = sweep_threshold(df.assign(_score=score), "_score")
        best = sweep.loc[sweep["f1"].idxmax()]
        pred = (score > float(best["threshold"])).astype(int)
        auc = roc_auc_score(df["is_attack"], score)
    else:
        pred = fuse_confidence_based(df, use_if=use_if, use_ae=use_ae)
        rule_score = confidence_rule_score(df, use_if=use_if, use_ae=use_ae)
        auc = roc_auc_score(df["is_attack"], rule_score)

    metrics = {
        "precision": precision_score(df["is_attack"], pred),
        "recall": recall_score(df["is_attack"], pred),
        "f1": f1_score(df["is_attack"], pred),
        "auc_roc": auc,
    }
    return metrics, pred

combo_definitions = {
    "IF + RF": {"isolation_forest", "random_forest"},
    "RF + AE": {"random_forest", "autoencoder"},
    "IF + RF + AE": {"isolation_forest", "random_forest", "autoencoder"},
}

val_ablation_rows = []
combo_preds = {}
for combo_name, members in combo_definitions.items():
    metrics, pred = evaluate_combo(fusion_df, members, PRIMARY_STRATEGY)
    combo_preds[combo_name] = pred
    val_ablation_rows.append({"combination": combo_name, **metrics})

single_model_rows = [
    {"combination": "IF alone", **recomputed_if},
    {"combination": "RF alone", **recomputed_rf},
    {"combination": "AE alone", **recomputed_ae},
]

val_ablation_df = pd.DataFrame(single_model_rows + val_ablation_rows)
val_ablation_df

## 7. Per-category recall — four-way (Isolation Forest / Random Forest / Autoencoder / Fusion)

In [ ]:
fusion_df["fusion_3model_pred"] = combo_preds["IF + RF + AE"]

attack_rows_fusion = fusion_df[fusion_df["Label"] != "BENIGN"].copy()
four_way_recall = (
    attack_rows_fusion.groupby("Label")[["if_pred", "rf_pred", "ae_pred", "fusion_3model_pred"]]
    .mean()
    .rename(columns={
        "if_pred": "isolation_forest_recall",
        "rf_pred": "random_forest_recall",
        "ae_pred": "autoencoder_recall",
        "fusion_3model_pred": "fusion_recall",
    })
)
four_way_recall["n_rows"] = attack_rows_fusion.groupby("Label").size()
four_way_recall = four_way_recall.sort_values("fusion_recall")
four_way_recall

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
categories_plot = four_way_recall.index.tolist()
x = np.arange(len(categories_plot))
width = 0.2

ax.bar(x - 1.5 * width, four_way_recall["isolation_forest_recall"], width, label="Isolation Forest", color="#4C72B0")
ax.bar(x - 0.5 * width, four_way_recall["random_forest_recall"], width, label="Random Forest", color="#DD8452")
ax.bar(x + 0.5 * width, four_way_recall["autoencoder_recall"], width, label="Autoencoder", color="#55A868")
ax.bar(x + 1.5 * width, four_way_recall["fusion_recall"], width, label="Fusion (IF+RF+AE)", color="#C44E52")

ax.set_xticks(x)
ax.set_xticklabels(categories_plot, rotation=45, ha="right")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.set_title("Per-category recall — IF vs RF vs AE vs Fusion (CICIDS-2017 val)")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Zero-day (LOAO) fusion evaluation

None of the three base notebooks saved row-level LOAO predictions, only aggregate
per-category recall/FPR -- so this section retrains Isolation Forest, Random Forest, and the
Autoencoder per fold (10 folds, same categories/data as 01/02/03), using each model's own
already-established hyperparameters verbatim (§1's config), to get row-level predictions to
fuse. This is the one deliberate exception to "don't retrain," agreed with the user given no
other way exists to answer the zero-day fusion question. `01_isolation_forest.ipynb`,
`02_random_forest.ipynb`, and `03_autoencoder.ipynb` themselves are not touched.

If `PRIMARY_STRATEGY` is `'confidence_based'`, no extra per-fold calibration is needed --
each model's own native fold-level output (Isolation Forest's `.predict()`, Random Forest's
`.predict_proba()`, the Autoencoder's own percentile-tuned fold threshold) is used directly.
If `'weighted_average'` instead, the fused threshold per fold is set at the same
`AE_CHOSEN_PERCENTILE` of *that fold's own zero-day test set's BENIGN rows'* fused score --
a mild transductive convenience specific to this LOAO extension (label-informed on the same
set being evaluated), disclosed here rather than treated as equivalent rigor to the
label-free calibration used everywhere else in this notebook.

**Runtime note**: this loop retrains a GPU Autoencoder per fold (the same ~300-400s/fold
cost measured in `03_autoencoder.ipynb`) plus an Isolation Forest and a SMOTE-resampled
Random Forest per fold -- expect roughly the same order of magnitude as the Autoencoder's
own LOAO loop (~1 hour), plus RF/IF's (smaller) additional cost.

In [ ]:
SENTINEL_COLS_LOAO = ["Init_Win_bytes_forward", "Init_Win_bytes_backward"]
NON_FEATURE_COLS_LOAO = ["Label"]

def select_raw_feature_columns(df):
    return [c for c in df.columns if c not in NON_FEATURE_COLS_LOAO and pd.api.types.is_numeric_dtype(df[c])]

def impute_sentinels_train_only(train_df_fold, test_df_fold, sentinel_cols=SENTINEL_COLS_LOAO):
    train_df_fold = train_df_fold.copy()
    test_df_fold = test_df_fold.copy()
    for c in sentinel_cols:
        median = train_df_fold[c].median(skipna=True)
        train_df_fold[c] = train_df_fold[c].fillna(median)
        test_df_fold[c] = test_df_fold[c].fillna(median)
    return train_df_fold, test_df_fold

def build_smote_sampling_strategy(y):
    counts = pd.Series(y).value_counts()
    majority_count = int(counts.max())
    strategy = {}
    for cls, n in counts.items():
        n = int(n)
        if n >= SMOTE_FLAT_CAP:
            continue
        target = min(majority_count, n * SMOTE_TARGET_MULTIPLIER, SMOTE_FLAT_CAP)
        if target > n:
            strategy[cls] = target
    return strategy

def apply_smote_loao(X, y):
    strategy = build_smote_sampling_strategy(y)
    if not strategy:
        return X, y, False
    min_class_n = min(int((pd.Series(y) == cls).sum()) for cls in strategy)
    k_neighbors = max(1, min(5, min_class_n - 1))
    smote = SMOTE(sampling_strategy=strategy, k_neighbors=k_neighbors, random_state=SMOTE_RANDOM_STATE)
    X_res, y_res = smote.fit_resample(X, y)
    return X_res, y_res, True


class Autoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dims, bottleneck_dim, activation="relu"):
        super().__init__()
        act_cls = {"relu": nn.ReLU, "tanh": nn.Tanh, "leaky_relu": nn.LeakyReLU}[activation]
        encoder_dims = [input_dim] + list(hidden_dims) + [bottleneck_dim]
        decoder_dims = [bottleneck_dim] + list(hidden_dims)[::-1] + [input_dim]

        encoder_layers = []
        for i in range(len(encoder_dims) - 1):
            encoder_layers.append(nn.Linear(encoder_dims[i], encoder_dims[i + 1]))
            if i < len(encoder_dims) - 2:
                encoder_layers.append(act_cls())
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        for i in range(len(decoder_dims) - 1):
            decoder_layers.append(nn.Linear(decoder_dims[i], decoder_dims[i + 1]))
            if i < len(decoder_dims) - 2:
                decoder_layers.append(act_cls())
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(self, x):
        return self.decoder(self.encoder(x))


def train_autoencoder(X_fit, X_internal_val, input_dim, config, verbose=False):
    model = Autoencoder(input_dim, config["hidden_dims"], config["bottleneck_dim"], config["activation"]).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
    criterion = nn.MSELoss()

    fit_tensor = torch.tensor(X_fit, dtype=torch.float32)
    internal_val_tensor = torch.tensor(X_internal_val, dtype=torch.float32).to(DEVICE)
    loader = DataLoader(TensorDataset(fit_tensor), batch_size=config["batch_size"], shuffle=True)

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    epochs_run = 0

    for epoch in range(config["epochs"]):
        model.train()
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(internal_val_tensor), internal_val_tensor).item()
        epochs_run = epoch + 1

        if verbose:
            print(f"    epoch {epochs_run:3d}/{config['epochs']}  val_loss={val_loss:.6f}")

        if val_loss < best_val_loss - 1e-6:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config["early_stopping_patience"]:
                break

    model.load_state_dict(best_state)
    return model, epochs_run


def compute_reconstruction_error(model, X):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        reconstructed = model(X_tensor)
        errors = torch.mean((reconstructed - X_tensor) ** 2, dim=1).cpu().numpy()
    return errors


CATEGORIES = [
    ("DoS Hulk", "dos_hulk"),
    ("PortScan", "portscan"),
    ("DDoS", "ddos"),
    ("DoS GoldenEye", "dos_goldeneye"),
    ("FTP-Patator", "ftp_patator"),
    ("SSH-Patator", "ssh_patator"),
    ("DoS slowloris", "dos_slowloris"),
    ("DoS Slowhttptest", "dos_slowhttptest"),
    ("Bot", "bot"),
    ("Web Attack - Brute Force", "web_attack_brute_force"),
]

print(f"LOAO helper functions defined. Will retrain Isolation Forest, Random Forest, and "
      f"Autoencoder per fold ({len(CATEGORIES)} folds) using each model's own already-established "
      f"hyperparameters (see config above), then combine with PRIMARY_STRATEGY='{PRIMARY_STRATEGY}'.")

In [ ]:
loao_fusion_results = []
_loao_loop_start = time.perf_counter()

for display_name, slug in CATEGORIES:
    _fold_start = time.perf_counter()
    loao_train_df = pd.read_parquet(LOAO_DIR / f"loao_{slug}_train.parquet")
    loao_test_df = pd.read_parquet(LOAO_DIR / f"loao_{slug}_zeroday_test.parquet")
    loao_train_df, loao_test_df = impute_sentinels_train_only(loao_train_df, loao_test_df)
    loao_feature_cols = select_raw_feature_columns(loao_train_df)

    y_fold_true = (loao_test_df["Label"] != "BENIGN").astype(int).to_numpy()

    X_if_train = loao_train_df[loao_feature_cols].to_numpy(dtype=np.float64)
    X_if_test = loao_test_df[loao_feature_cols].to_numpy(dtype=np.float64)
    if_fold_model = IsolationForest(
        n_estimators=IF_N_ESTIMATORS, contamination=IF_CONTAMINATION,
        random_state=RANDOM_STATE, n_jobs=-1,
    )
    if_fold_model.fit(X_if_train)
    if_fold_pred = (if_fold_model.predict(X_if_test) == -1).astype(int)
    if_fold_anomaly_score = -if_fold_model.score_samples(X_if_test)

    y_rf_train = loao_train_df["Label"].to_numpy()
    X_rf_train_res, y_rf_train_res, smote_applied = apply_smote_loao(X_if_train, y_rf_train)
    rf_fold_model = RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, class_weight=RF_CLASS_WEIGHT,
        random_state=RANDOM_STATE, n_jobs=RF_N_JOBS,
    )
    rf_fold_model.fit(X_rf_train_res, y_rf_train_res)
    rf_fold_pred_label = rf_fold_model.predict(X_if_test)
    rf_fold_pred = (rf_fold_pred_label != "BENIGN").astype(int)
    benign_col_idx = list(rf_fold_model.classes_).index("BENIGN")
    rf_fold_proba = rf_fold_model.predict_proba(X_if_test)
    rf_fold_attack_probability = 1.0 - rf_fold_proba[:, benign_col_idx]

    fold_scaler = MinMaxScaler()
    fold_scaler.fit(loao_train_df[loao_feature_cols])
    loao_train_scaled = loao_train_df.copy()
    loao_test_scaled = loao_test_df.copy()
    loao_train_scaled[loao_feature_cols] = fold_scaler.transform(loao_train_df[loao_feature_cols])
    loao_test_scaled[loao_feature_cols] = fold_scaler.transform(loao_test_df[loao_feature_cols])

    fold_benign_train_df = loao_train_scaled[loao_train_scaled["Label"] == "BENIGN"]
    fold_benign_fit_df, fold_benign_internal_val_df = train_test_split(
        fold_benign_train_df, test_size=AE_BENIGN_INTERNAL_VAL_FRAC, random_state=RANDOM_STATE
    )
    X_ae_fit = fold_benign_fit_df[loao_feature_cols].to_numpy(dtype=np.float32)
    X_ae_internal_val = fold_benign_internal_val_df[loao_feature_cols].to_numpy(dtype=np.float32)
    ae_fold_model, ae_epochs_run = train_autoencoder(
        X_ae_fit, X_ae_internal_val, input_dim=len(loao_feature_cols), config=AE_CONFIG, verbose=False
    )
    X_ae_test = loao_test_scaled[loao_feature_cols].to_numpy(dtype=np.float32)
    ae_fold_test_errors = compute_reconstruction_error(ae_fold_model, X_ae_test)
    ae_fold_internal_val_errors = compute_reconstruction_error(ae_fold_model, X_ae_internal_val)
    ae_fold_threshold = float(np.percentile(ae_fold_internal_val_errors, AE_CHOSEN_PERCENTILE))
    ae_fold_pred = (ae_fold_test_errors > ae_fold_threshold).astype(int)

    fold_scores_df = pd.DataFrame({
        "if_pred": if_fold_pred,
        "rf_pred": rf_fold_pred,
        "ae_pred": ae_fold_pred,
        "if_norm": minmax_normalize(if_fold_anomaly_score),
        "ae_norm": minmax_normalize(ae_fold_test_errors),
        "rf_norm": rf_fold_attack_probability,
        "rf_attack_probability": rf_fold_attack_probability,
        "is_attack": y_fold_true,
    })

    def fold_recall_fpr(pred):
        pred = np.asarray(pred)
        tp = int(((pred == 1) & (y_fold_true == 1)).sum())
        fn = int(((pred == 0) & (y_fold_true == 1)).sum())
        fp = int(((pred == 1) & (y_fold_true == 0)).sum())
        tn = int(((pred == 0) & (y_fold_true == 0)).sum())
        recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        fpr = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
        return recall, fpr

    fold_combo_preds = {}
    for combo_name, members in combo_definitions.items():
        use_if = "isolation_forest" in members
        use_ae = "autoencoder" in members
        if PRIMARY_STRATEGY == "weighted_average":
            weights = {k: v for k, v in FUSION_WEIGHTS.items() if k in members}
            score = fuse_weighted_average(fold_scores_df, weights)
            benign_scores = score[y_fold_true == 0]
            threshold = (
                float(np.percentile(benign_scores, AE_CHOSEN_PERCENTILE))
                if len(benign_scores) else float(np.median(score))
            )
            fold_combo_preds[combo_name] = (score > threshold).astype(int)
        else:
            fold_combo_preds[combo_name] = fuse_confidence_based(
                fold_scores_df, use_if=use_if, use_ae=use_ae
            ).to_numpy()

    fold_seconds = time.perf_counter() - _fold_start

    row = {
        "category": display_name,
        "n_test_rows": len(loao_test_df),
        "n_attack_rows": int(y_fold_true.sum()),
        "n_benign_rows": int((y_fold_true == 0).sum()),
        "fold_seconds": fold_seconds,
        "ae_epochs_run": ae_epochs_run,
        "smote_applied": smote_applied,
    }
    row["isolation_forest_recall"], row["isolation_forest_fpr"] = fold_recall_fpr(if_fold_pred)
    row["random_forest_recall"], row["random_forest_fpr"] = fold_recall_fpr(rf_fold_pred)
    row["autoencoder_recall"], row["autoencoder_fpr"] = fold_recall_fpr(ae_fold_pred)
    for combo_name in combo_definitions:
        key = combo_name.lower().replace(" + ", "_")
        recall, fpr = fold_recall_fpr(fold_combo_preds[combo_name])
        row[f"{key}_recall"] = recall
        row[f"{key}_fpr"] = fpr

    loao_fusion_results.append(row)
    print(f"{display_name:30s}  IF={row['isolation_forest_recall']:.3f}  RF={row['random_forest_recall']:.3f}  "
          f"AE={row['autoencoder_recall']:.3f}  IF+RF={row['if_rf_recall']:.3f}  RF+AE={row['rf_ae_recall']:.3f}  "
          f"IF+RF+AE={row['if_rf_ae_recall']:.3f}  time={fold_seconds:.1f}s")

loao_loop_seconds = time.perf_counter() - _loao_loop_start
print(f"\nFull LOAO fusion loop ({len(CATEGORIES)} folds) wall-clock time: {loao_loop_seconds:.2f}s (AE device={DEVICE})")

loao_fusion_df = pd.DataFrame(loao_fusion_results)

In [ ]:
loao_fusion_df

## 9. Overall ablation summary table

The key thesis artifact: one row per configuration, val-set Precision/Recall/F1/AUC-ROC
alongside the mean LOAO recall/FPR across the 10 zero-day categories (unweighted mean --
per-category prevalence varies too much to weight meaningfully, same caveat as the base
notebooks' own LOAO sections).

In [ ]:
overall_rows = []

for name, val_metrics, recall_col, fpr_col in [
    ("IF alone", recomputed_if, "isolation_forest_recall", "isolation_forest_fpr"),
    ("RF alone", recomputed_rf, "random_forest_recall", "random_forest_fpr"),
    ("AE alone", recomputed_ae, "autoencoder_recall", "autoencoder_fpr"),
]:
    overall_rows.append({
        "combination": name,
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
        "val_auc_roc": val_metrics["auc_roc"],
        "loao_mean_recall": loao_fusion_df[recall_col].mean(),
        "loao_mean_fpr": loao_fusion_df[fpr_col].mean(),
    })

for combo_name in combo_definitions:
    key = combo_name.lower().replace(" + ", "_")
    val_row = val_ablation_df.set_index("combination").loc[combo_name]
    overall_rows.append({
        "combination": combo_name,
        "val_precision": val_row["precision"],
        "val_recall": val_row["recall"],
        "val_f1": val_row["f1"],
        "val_auc_roc": val_row["auc_roc"],
        "loao_mean_recall": loao_fusion_df[f"{key}_recall"].mean(),
        "loao_mean_fpr": loao_fusion_df[f"{key}_fpr"].mean(),
    })

overall_ablation_df = pd.DataFrame(overall_rows)
overall_ablation_df

In [ ]:
ablation_summary_path = REPORTS_DIR / "ensemble_ablation_summary.csv"
overall_ablation_df.to_csv(ablation_summary_path, index=False)
print("Saved ablation summary to", ablation_summary_path)

## 10. Save outputs

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

fusion_layer_code = f'''\"\"\"Fusion Layer combination logic, generated by 04_fusion_layer.ipynb.

Combines Isolation Forest, Random Forest, and Autoencoder outputs into a single ensemble
decision. Primary strategy: {PRIMARY_STRATEGY!r} (see the notebook's Section 5.1 for why this
was chosen over the alternative weighted-average strategy).
\"\"\"

FUSION_WEIGHTS = {FUSION_WEIGHTS!r}
RF_CONFIDENCE_THRESHOLD = {RF_CONFIDENCE_THRESHOLD}
PRIMARY_STRATEGY = {PRIMARY_STRATEGY!r}


def fuse_weighted_average(if_norm, rf_norm, ae_norm, weights=FUSION_WEIGHTS):
    total_w = sum(weights.values())
    return (
        weights.get("isolation_forest", 0) / total_w * if_norm
        + weights.get("random_forest", 0) / total_w * rf_norm
        + weights.get("autoencoder", 0) / total_w * ae_norm
    )


def fuse_confidence_based(if_pred, rf_attack_probability, ae_pred,
                           rf_confidence_threshold=RF_CONFIDENCE_THRESHOLD):
    \"\"\"1 = flagged as attack, 0 = flagged as normal.

    if_pred / ae_pred: each model's own already-thresholded binary flag.
    rf_attack_probability: Random Forest's 1 - P(BENIGN), continuous in [0, 1].
    \"\"\"
    rf_flag = rf_attack_probability > rf_confidence_threshold
    unsupervised_agree = bool(if_pred) and bool(ae_pred)
    return int(rf_flag or unsupervised_agree)
'''

fusion_layer_path = MODELS_DIR / "fusion_layer.py"
with open(fusion_layer_path, "w", encoding="utf-8") as f:
    f.write(fusion_layer_code)
print("Saved fusion logic to", fusion_layer_path)

fusion_config = {
    "primary_strategy": PRIMARY_STRATEGY,
    "fusion_weights": FUSION_WEIGHTS,
    "rf_confidence_threshold": RF_CONFIDENCE_THRESHOLD,
    "strategy_a_val_metrics": strategy_a_metrics,
    "strategy_b_val_metrics": strategy_b_metrics,
    "strategy_a_threshold": strategy_a_threshold,
}
fusion_config_path = MODELS_DIR / "fusion_layer_config.json"
with open(fusion_config_path, "w", encoding="utf-8") as f:
    json.dump(fusion_config, f, indent=2)
print("Saved config to", fusion_config_path)

In [ ]:
fusion_scores_val_df = pd.DataFrame({
    "row_index": val_df.index.to_numpy(),
    "if_norm": fusion_df["if_norm"].to_numpy(),
    "rf_norm": fusion_df["rf_norm"].to_numpy(),
    "ae_norm": fusion_df["ae_norm"].to_numpy(),
    "weighted_avg_score": fusion_df["weighted_avg_score"].to_numpy(),
    "confidence_rule_score": fusion_df["confidence_rule_score"].to_numpy(),
    "prediction": combo_preds["IF + RF + AE"],
})

fusion_scores_path = DATA_FEATURES_DIR / "cicids2017_fusion_scores_val.parquet"
fusion_scores_val_df.to_parquet(fusion_scores_path, index=False)
print("Saved fused val-set scores to", fusion_scores_path)
fusion_scores_val_df.head()

## Interpretation notes

Read `overall_ablation_df` (§9) as the primary answer to this notebook's motivating
question: compare the `IF + RF + AE` row's `val_f1`/`loao_mean_recall` against `IF + RF`
and `RF + AE`. If the 3-model row does not clearly beat both 2-model rows, that's evidence
Autoencoder does not earn its place given its high correlation with Isolation Forest found
in `03_autoencoder.ipynb` (0.925 val / 0.977 LOAO) — a simpler 2-model fusion would be an
equally defensible final architecture. If it does clearly beat both, that's evidence the
correlation, while high, still leaves enough residual disagreement between Isolation Forest
and Autoencoder for the fusion to exploit.